Section C | PySpark DataFrame Operations

11. Load the meter readings and zone master Delta tables into separate PySpark DataFrames; print their
schemas.
12. Use select(), filter() and orderBy() to display high-flow Valid readings.
13. Add a flow_category column: High for flow_lpm >= 60, Medium for 30 to below 60, Low otherwise;
handle nulls explicitly.
14. Join the DataFrames and calculate average flow_lpm for each zone_type.
15. Use PySpark groupBy() and agg() to identify the three zones with the highest valid water consumption.

In [0]:

spark.sql("USE CATALOG workspace1")
spark.sql("USE SCHEMA water_operations")

DataFrame[]

In [0]:
readings_df = spark.table("water_meter_readings_source")
zones_df = spark.table("water_zones")

readings_df.printSchema()
zones_df.printSchema()

root
 |-- reading_id: integer (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- reading_time: timestamp (nullable = true)
 |-- flow_lpm: double (nullable = true)
 |-- pressure_bar: double (nullable = true)
 |-- chlorine_mg_l: double (nullable = true)
 |-- consumption_liters: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_system: string (nullable = true)

root
 |-- zone_id: string (nullable = true)
 |-- zone_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- zone_type: string (nullable = true)
 |-- daily_capacity_liters: long (nullable = true)
 |-- active: boolean (nullable = true)
 |-- commissioned_on: string (nullable = true)



In [0]:
from pyspark.sql.functions import when,col

readings_df = readings_df.withColumn(
    "flow_category",
    when(col("flow_lpm").isNull(), "Unknown")
    .when(col("flow_lpm") >= 60, "High")
    .when(col("flow_lpm") >= 30, "Medium")
    .otherwise("Low")
)

readings_df.select(
    "reading_id",
    "flow_lpm",
    "flow_category"
).show()

+----------+--------+-------------+
|reading_id|flow_lpm|flow_category|
+----------+--------+-------------+
|      1001|    39.1|       Medium|
|      1002|    59.9|       Medium|
|      1003|    25.7|          Low|
|      1004|    56.7|       Medium|
|      1005|    61.8|         High|
|      1006|    35.8|       Medium|
|      1007|    65.0|         High|
|      1008|    70.3|         High|
|      1009|    30.8|       Medium|
|      1010|    81.3|         High|
|      1011|    13.0|          Low|
|      1012|    36.3|       Medium|
|      1013|    80.7|         High|
|      1014|    22.4|          Low|
|      1015|    30.9|       Medium|
|      1016|    12.9|          Low|
|      1017|    28.0|          Low|
|      1018|    70.5|         High|
|      1019|    23.0|          Low|
|      1020|    67.7|         High|
+----------+--------+-------------+
only showing top 20 rows


In [0]:
joined_df = readings_df.join(
    zones_df,
    readings_df.zone_id == zones_df.zone_id,
    "inner"
)

zone_type_avg = joined_df.groupBy(
    zones_df.zone_type
).avg("flow_lpm")

zone_type_avg.show()

+-------------+------------------+
|    zone_type|     avg(flow_lpm)|
+-------------+------------------+
|   Commercial| 56.10000000000001|
|  Residential|45.155737704918025|
|        Mixed|         38.165625|
|   Industrial| 43.14374999999999|
|Institutional|          49.33125|
+-------------+------------------+



In [0]:
from pyspark.sql.functions import sum, desc

top_zones = readings_df.filter(
    (col("reading_status") == "Valid") &
    (col("consumption_liters") > 0)
).groupBy(
    "zone_id"
).agg(
    sum("consumption_liters").alias("total_consumption")
).orderBy(
    desc("total_consumption")
).limit(3)

top_zones.show()

+-------+-----------------+
|zone_id|total_consumption|
+-------+-----------------+
|    W03|            35665|
|    W08|            33689|
|    W05|            28717|
+-------+-----------------+



Section D | Data Quality and Cleaning

16. Identify duplicate reading_id values and display every row associated with those IDs.
17. Find missing pressure_bar and source_system values and count the affected records.
18. Identify invalid measurements: negative flow_lpm, nonpositive consumption_liters or missing zone_id.
19. Create water_readings_clean as a new Delta table containing one row per reading_id, with Valid status
and positive measurements; document the duplicate-selection rule.
20. Use the service request data to identify zones with High-priority requests and compare their average
recorded pressure

In [0]:
from pyspark.sql.functions import count

duplicate_ids = readings_df.groupBy("reading_id").agg(
    count("*").alias("id_count")
).filter(
    col("id_count") > 1
)

duplicate_ids.show()

duplicate_rows = readings_df.join(
    duplicate_ids.select("reading_id"),
    "reading_id",
    "inner"
)

duplicate_rows.orderBy("reading_id").show(truncate=False)

+----------+--------+
|reading_id|id_count|
+----------+--------+
|      1044|       2|
+----------+--------+

+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+-------------+
|reading_id|meter_id|zone_id|reading_time       |flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|flow_category|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+-------------+
|1044      |M014    |W04    |2026-10-05 10:10:00|79.3    |1.95        |1.23         |2661              |Valid         |SmartMeter   |High         |
|1044      |M014    |W04    |2026-10-16 13:00:00|36.0    |3.2         |0.8          |1300              |Valid         |Gateway      |Medium       |
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+-------------+



In [0]:
missing_values = readings_df.filter(
    col("pressure_bar").isNull() |
    col("source_system").isNull()
)

missing_values.show(truncate=False)

missing_values.select(
    count("*").alias("affected_records")
).show()

+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+-------------+
|reading_id|meter_id|zone_id|reading_time       |flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|flow_category|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+-------------+
|1153      |M020    |W10    |2026-10-16 10:00:00|31.4    |NULL        |0.71         |1020              |Valid         |Gateway      |Medium       |
|1155      |M006    |W06    |2026-10-16 12:00:00|44.0    |2.9         |2.95         |1560              |Suspect       |NULL         |Medium       |
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+-------------+

+----------------+
|affected_records|
+----------------+
|               2|
+----------------+



In [0]:
invalid_readings = readings_df.filter(
    (col("flow_lpm") < 0) |
    (col("consumption_liters") <= 0) |
    (col("zone_id").isNull())
)

invalid_readings.show(truncate=False)

print("Invalid reading count:", invalid_readings.count())

+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+-------------+
|reading_id|meter_id|zone_id|reading_time       |flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|flow_category|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+-------------+
|1151      |M004    |W04    |2026-10-16 08:00:00|-12.5   |2.8         |0.62         |500               |Suspect       |Gateway      |Low          |
|1152      |M015    |W05    |2026-10-16 09:00:00|22.0    |2.1         |0.5          |0                 |Valid         |SmartMeter   |Low          |
|1156      |M012    |NULL   |2026-10-16 12:30:00|28.0    |3.1         |0.52         |870               |Valid         |Gateway      |Low          |
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

valid_readings = readings_df.filter(
    (col("reading_status") == "Valid") &
    (col("flow_lpm") > 0) &
    (col("consumption_liters") > 0) &
    (col("zone_id").isNotNull())
)

window_spec = Window.partitionBy("reading_id").orderBy(
    col("reading_time").asc_nulls_last(),
    col("meter_id").asc_nulls_last()
)

water_readings_clean = valid_readings.withColumn(
    "row_num",
    row_number().over(window_spec)
).filter(
    col("row_num") == 1
).drop("row_num")

water_readings_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("water_readings_clean")

In [0]:
clean_df = spark.table("water_readings_clean")

print("Clean row count:", clean_df.count())
print("Unique reading IDs:", clean_df.select("reading_id").distinct().count())

clean_df.show()

Clean row count: 134
Unique reading IDs: 134
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+-------------+
|reading_id|meter_id|zone_id|       reading_time|flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|flow_category|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+-------------+
|      1001|    M001|    W01|2026-10-01 07:00:00|    39.1|        3.56|         0.75|              2365|         Valid|      Gateway|       Medium|
|      1002|    M002|    W02|2026-10-01 08:10:00|    59.9|         4.6|         0.39|              3545|         Valid|   SmartMeter|       Medium|
|      1003|    M003|    W03|2026-10-01 09:20:00|    25.7|        2.22|         1.15|               548|         Valid|     FieldApp|          Low|
|      1004|    M004|    W04|2026-10-01 10:00:00|    56.7|        2

In [0]:
from pyspark.sql.functions import avg
requests_df = spark.table("water_service_requests")

high_priority_zones = requests_df.filter(
    col("priority") == "High"
).select(
    "zone_id"
).distinct()

high_priority_pressure = readings_df.join(
    high_priority_zones,
    "zone_id",
    "inner"
).groupBy(
    "zone_id"
).agg(avg("pressure_bar").alias("average_pressure"),
    count("*").alias("reading_count")
).orderBy("zone_id")

high_priority_pressure.show()

+-------+------------------+-------------+
|zone_id|  average_pressure|reading_count|
+-------+------------------+-------------+
|    W01| 3.437333333333333|           15|
|    W02|3.3573333333333335|           15|
|    W03|             3.178|           15|
|    W04|3.2829411764705885|           17|
|    W05|               3.5|           16|
|    W06|3.2250000000000005|           16|
|    W07|3.3706666666666667|           15|
|    W08|              2.92|           15|
|    W09|3.6040000000000005|           15|
|    W10|3.7079999999999997|           16|
+-------+------------------+-------------+



40. Write a DataFrame to Parquet in an authorized Unity Catalog Volume, then load that Parquet dataset
into a new managed Delta table

In [0]:
readings_df = spark.table("water_meter_readings_work")
readings_df.show(5)

+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|reading_id|meter_id|zone_id|       reading_time|flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|      1001|    M001|    W01|2026-10-01 07:00:00|    39.1|        3.56|         0.75|              2365|         Valid|      Gateway|
|      1002|    M002|    W02|2026-10-01 08:10:00|    59.9|         4.6|         0.39|              3545|         Valid|   SmartMeter|
|      1003|    M003|    W03|2026-10-01 09:20:00|    25.7|        2.22|         1.15|               548|         Valid|     FieldApp|
|      1004|    M004|    W04|2026-10-01 10:00:00|    56.7|        2.84|          1.1|              2871|         Valid|      Gateway|
|      1006|    M006|    W06|2026-10-01 12:20:00|    35.8|    

In [0]:
parquet_path = "/Volumes/workspace1/water_operations/water_data/reading_parquet_output"

readings_df.write.mode("overwrite").parquet(parquet_path)

In [0]:
parquet_df = spark.read.parquet(parquet_path)

parquet_df.show(5)

+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|reading_id|meter_id|zone_id|       reading_time|flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|      1001|    M001|    W01|2026-10-01 07:00:00|    39.1|        3.56|         0.75|              2365|         Valid|      Gateway|
|      1002|    M002|    W02|2026-10-01 08:10:00|    59.9|         4.6|         0.39|              3545|         Valid|   SmartMeter|
|      1003|    M003|    W03|2026-10-01 09:20:00|    25.7|        2.22|         1.15|               548|         Valid|     FieldApp|
|      1004|    M004|    W04|2026-10-01 10:00:00|    56.7|        2.84|          1.1|              2871|         Valid|      Gateway|
|      1006|    M006|    W06|2026-10-01 12:20:00|    35.8|    

In [0]:
parquet_df.write.format("delta").mode("overwrite").saveAsTable(
    "water_readings_from_parquet"
)

In [0]:
new_df = spark.table("water_readings_from_parquet")

print("Total rows:", new_df.count())
new_df.show(5)

Total rows: 163
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|reading_id|meter_id|zone_id|       reading_time|flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|      1001|    M001|    W01|2026-10-01 07:00:00|    39.1|        3.56|         0.75|              2365|         Valid|      Gateway|
|      1002|    M002|    W02|2026-10-01 08:10:00|    59.9|         4.6|         0.39|              3545|         Valid|   SmartMeter|
|      1003|    M003|    W03|2026-10-01 09:20:00|    25.7|        2.22|         1.15|               548|         Valid|     FieldApp|
|      1004|    M004|    W04|2026-10-01 10:00:00|    56.7|        2.84|          1.1|              2871|         Valid|      Gateway|
|      1006|    M006|    W06|2026-10-01 12:20:

Section I | Structured Streaming

41. Create a streaming DataFrame using the built-in rate source with five rows per second.
42. Transform the generated value into a simulated meter_id and flow_lpm column.
43. Filter the stream to retain simulated flow_lpm above a threshold of your choice.
Page 3
Azure Databricks  |  City Water Network Monitoring
44. Aggregate streaming records by a derived flow category and display the live counts using a supported
sink.
45. Stop the running streaming query and explain the role of a checkpoint for durable streaming output

In [0]:
rate_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

rate_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



In [0]:
from pyspark.sql.functions import col, concat, lit

stream_df = rate_df.withColumn(
    "meter_id",
    concat(lit("MTR_"), col("value").cast("string"))
).withColumn(
    "flow_lpm",
    ((col("value") % 100) + 1).cast("double")
)

In [0]:
filtered_stream_df = stream_df.filter(
    col("flow_lpm") > 50
)

In [0]:
from pyspark.sql.functions import when, count

categorized_stream_df = filtered_stream_df.withColumn(
    "flow_category",
    when(col("flow_lpm") >= 75, "High").otherwise("Medium")
)

flow_counts_df = categorized_stream_df.groupBy(
    "flow_category"
).agg(
    count("*").alias("total_readings")
)

In [0]:
checkpoint_path = "/Volumes/workspace1/water_operations/water_data/flow_checkpoint"

stream_query = flow_counts_df.writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .start()

In [0]:
print("Is stream active:", stream_query.isActive)

Is stream active: False


In [0]:
stream_query.stop()